In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime, date
import ast

In [2]:
def ensure_columns(df, fill_value=pd.NA):

    required_columns = ['json_name', 'column_name', 'path',
                        'list_path', 'subfield_path', 'var_type', 
                        'data_type', 'file_path']
    
    for col in required_columns:
        if col not in df.columns:
            df[col] = fill_value

    return df

In [3]:
def combine_list(df):

    df['final_path'] = None

    for ix, row in df.iterrows():
        if row['platform'] == 'Tiktok':
            final_path =  row['path']
            
            if isinstance(final_path, str):
                final_path = ast.literal_eval(final_path)
               
            
        else:
            file_list = row['file_path'].split('/')
            path_list = row['path']
        
            if pd.isna(path_list):
                final_path = row['file_path']
            else:
                if isinstance(path_list, str):
                    path_list = ast.literal_eval(path_list)

                if isinstance(path_list, list):
                    final_path = file_list + path_list
                    path = '/'.join(path_list)
                    df.at[ix, 'path'] = path

        #print(final_path)
        final_path = '/'.join(final_path)
        df.at[ix, 'final_path'] = final_path 
              
    
    return df
    


In [4]:

#ROOT = '/home/rvissche/GIT/social-media-data-map'
ROOT = '/home/bsc/bsc093754/GIT/social-media-data-map'
root_dir = Path(f"{ROOT}/data/raw/platform_data")

save_dir = f'{ROOT}/data/processed'
  

def create_dataset(root_dir, save_dir):

    dfs_list = []
    for platform_dir in root_dir.iterdir():
        for output_dir in platform_dir.rglob("Output"):

            if output_dir.is_dir():

                for csv_file in output_dir.glob("*.csv"):

                    try:
                        file_name =  csv_file.stem
                        
                        
                        df = pd.read_csv(csv_file)
                        df = ensure_columns(df)


                        df['participant'] = file_name
                        col = df.pop('participant') 
                        df.insert(0, 'participant', col)  

                        if 'TikTok' in str(csv_file):
                            df['platform'] = 'Tiktok'
                        if 'Instagram' in str(csv_file):
                            df['platform'] = 'Instagram'
                        if 'Facebook' in str(csv_file):
                            df['platform'] = 'Facebook'
                        if 'Youtube' in str(csv_file):
                            df['platform'] = 'Youtube'
                        if 'Twitter' in str(csv_file):
                            df['platform'] = 'Twitter'

                        
                        col = df.pop('platform') 
                        df.insert(1, 'platform', col) 
                        df = combine_list(df)
                        
                        dfs_list.append(df)
                    
                    except Exception as e:
                        print(f"Failed to load {csv_file}: {e}")

    print(dfs_list)
    dfs = pd.concat(dfs_list, ignore_index=True)
    print('DATASET CREATED at ', datetime.now())
    dfs.to_csv(f'{save_dir}/participant_data.csv')
    print('DATASET SAVED TO ', save_dir)

    return dfs





In [5]:
create_dataset(root_dir, save_dir)

[                               participant platform    column_name  \
0    Output_TT_structure_tiktok_takeout_es   Tiktok            App   
1    Output_TT_structure_tiktok_takeout_es   Tiktok            App   
2    Output_TT_structure_tiktok_takeout_es   Tiktok     IsFastLane   
3    Output_TT_structure_tiktok_takeout_es   Tiktok            App   
4    Output_TT_structure_tiktok_takeout_es   Tiktok     IsFastLane   
..                                     ...      ...            ...   
126  Output_TT_structure_tiktok_takeout_es   Tiktok   AddYoursText   
127  Output_TT_structure_tiktok_takeout_es   Tiktok    Description   
128  Output_TT_structure_tiktok_takeout_es   Tiktok           Name   
129  Output_TT_structure_tiktok_takeout_es   Tiktok       Platform   
130  Output_TT_structure_tiktok_takeout_es   Tiktok  Profile Photo   

                                                  path  \
0               ['Activity', 'Favorite Videos', 'App']   
1                 ['Activity', 'Follower L

,participant,platform,column_name,path,list_path,subfield_path,var_type,data_type,json_name,file_path,final_path
0,Output_TT_structure_tiktok_takeout_es,Tiktok,App,"['Activity', 'Favorite Videos', 'App']",[],"['Activity', 'Favorite Videos', 'App']",static,number,<NA>,<NA>,Activity/Favorite Videos/App
1,Output_TT_structure_tiktok_takeout_es,Tiktok,App,"['Activity', 'Follower List', 'App']",[],"['Activity', 'Follower List', 'App']",static,number,<NA>,<NA>,Activity/Follower List/App
2,Output_TT_structure_tiktok_takeout_es,Tiktok,IsFastLane,"['Activity', 'Follower List', 'IsFastLane']",[],"['Activity', 'Follower List', 'IsFastLane']",static,boolean,<NA>,<NA>,Activity/Follower List/IsFastLane
3,Output_TT_structure_tiktok_takeout_es,Tiktok,App,"['Activity', 'Following List', 'App']",[],"['Activity', 'Following List', 'App']",static,number,<NA>,<NA>,Activity/Following List/App
4,Output_TT_structure_tiktok_takeout_es,Tiktok,IsFastLane,"['Activity', 'Following List', 'IsFastLane']",[],"['Activity', 'Following List', 'IsFastLane']",static,boolean,<NA>,<NA>,Activity/Following List/IsFastLane
...,...,...,...,...,...,...,...,...,...,...,...
15809,Output_structure_YouTube_090a8476865a56db,Youtube,products,products,[],['products'],list,string,historial-de-reproducciones.json,Takeout/YouTube y YouTube Music/historial/hist...,Takeout/YouTube y YouTube Music/historial/hist...
15810,Output_structure_YouTube_090a8476865a56db,Youtube,activityControls,activityControls,[],['activityControls'],list,string,historial-de-reproducciones.json,Takeout/YouTube y YouTube Music/historial/hist...,Takeout/YouTube y YouTube Music/historial/hist...
15811,Output_structure_YouTube_090a8476865a56db,Youtube,name,details/name,['details'],['name'],list,string,historial-de-búsqueda.json,Takeout/YouTube y YouTube Music/historial/hist...,Takeout/YouTube y YouTube Music/historial/hist...
15812,Output_structure_YouTube_090a8476865a56db,Youtube,name,subtitles/name,['subtitles'],['name'],list,string,historial-de-reproducciones.json,Takeout/YouTube y YouTube Music/historial/hist...,Takeout/YouTube y YouTube Music/historial/hist...
